In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.2"


import matplotlib.pyplot as plt
import numpy as np

from openpi.policies import policy_config as _policy_config
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

In [ ]:
config = _config.get_config("pi05_agilex_rtc")
checkpoint_dir = "../checkpoints/pi05_agilex_rtc_ada/pi05_rtc_ada_pick_beverage_1218/49999"

In [ ]:
# 2. Create the specific DataConfig from the factory defined in your main config.
#    This step prepares all dataset-related settings, including transforms.
data_config = config.data.create(config.assets_dirs, config.model)

# 3. Create the base dataset. This function will use the `repo_id` from
#    the data_config to load the LeRobot dataset from the hub or a local path.
base_dataset = _data_loader.create_torch_dataset(data_config, config.model.action_horizon, config.model)

# 4. Wrap the base dataset with the transforms defined in your config.
#    This applies the repack, data, and model transforms.
dataset = _data_loader.TransformedDataset(
    base_dataset,
    [
        *data_config.repack_transforms.inputs,
        # *data_config.data_transforms.inputs, # test data feed to model
        # *data_config.model_transforms.inputs, # test data feed to model
    ],
)

In [ ]:
n_steps = 10
policy = _policy_config.create_trained_policy(
    config,
    checkpoint_dir,
    sample_kwargs={"num_steps": n_steps, "infer_time_schedule": "adaptive", "C": 1, "alpha": 0.6, "T_min": 0.1},
)

In [ ]:
from tqdm import tqdm

max_inferences = 20
all_gt_actions = []
all_infer_actions = []
all_x_ts = []
all_v_ts = []
all_targets = []
delay = 5

streaming = False

for i in tqdm(range(max_inferences)):
    data = dataset[i * config.model.action_horizon]  # i * chunk_size
    if i >= max_inferences:
        break

    gt_actions = data["actions"].numpy().copy()  # Shape: (50, 14)
    # del data["actions"]

    if delay > 0:
        data["action_prefix"] = gt_actions[:delay].copy()
        data["delay"] = np.array(delay)

    data["debug"] = True

    if streaming:
        stream_actions = [gt_actions[:delay]]

        def on_actions_ready(actions):
            stream_actions.append(actions)

        outputs = policy.infer_streaming(data, on_actions_ready=on_actions_ready)
        infer_actions = np.concatenate(stream_actions, axis=0)
    else:
        outputs = policy.infer(data)
        infer_actions = outputs["actions"]  # Shape: (50, 14)

    all_infer_actions.append(infer_actions)
    all_gt_actions.append(gt_actions)
    # all_x_ts.append(outputs["all_x_t"])
    # all_v_ts.append(outputs["all_v_t"])
    # all_targets.append(outputs["targets"])

In [ ]:
all_x_ts = np.stack(all_x_ts)
x_t = all_x_ts[:, :, 0, delay:, 7:14]
print(x_t.shape)

all_v_ts = np.stack(all_v_ts)
v_t = all_v_ts[:, :, 0, delay:, 7:14]
print(v_t.shape)

# np.save(f"x_t_s{n_steps}.npy", x_t)

all_targets = np.stack(all_targets)
targets = all_targets[:, :, delay:, 7:14]
print(targets.shape)

## Diff analysis

In [ ]:
def metric_diff(x, m):
    if m == "l2":
        return np.linalg.norm(np.diff(x, axis=1), axis=-1)
    elif m == "rel_l2":
        return np.linalg.norm(np.diff(x, axis=1), axis=-1) / (np.linalg.norm(x[:, :-1], axis=-1) + 1e-8)
    elif m == "l1":
        return np.sum(np.abs(np.diff(x, axis=1)), axis=-1)
    elif m == "rel_l1":
        return np.sum(np.abs(np.diff(x, axis=1)), axis=-1) / (np.sum(np.abs(x[:, :-1]), axis=-1) + 1e-8)
    else:
        raise ValueError(f"Unknown metric: {m}")


x_t_diff = metric_diff(x_t, "l2")
print(x_t_diff.shape)  # (n_rollouts, n_steps, horizon)

x_t_diff_avg = np.mean(x_t_diff, axis=(0))
print(x_t_diff_avg.shape)

plt.imshow(x_t_diff_avg)
# for h in [0, 10, 20, 30]:
#     plt.plot(x_t_diff_avg[:, h], label=f"h={h}")
# plt.legend()
plt.show()

## Final analysis

In [ ]:
def metric_final(x, final, m):
    if m == "l2":
        return np.linalg.norm(x[:, 1:] - final, axis=-1)
    elif m == "l1":
        return np.sum(np.abs(x[:, 1:] - final), axis=-1)
    elif m == "mse":
        # Normalized MSE to Final Prediction
        # $$E_{t, i} = \frac{\| x_{t, i} - x_{final, i} \|_2}{\| x_{init, i} - x_{final, i} \|_2}$$
        return np.sum((x[:, 1:] - final) ** 2, axis=-1) / np.sum((x[:, 0:1] - final) ** 2, axis=-1)
    else:
        raise ValueError(f"Unknown metric: {m}")


# x_t_final = x_t[:, -1:]
x_t_final = targets
diff_final = metric_final(x_t, x_t_final, "mse")
print(diff_final.shape)

diff_final_avg = np.mean(diff_final, axis=(0))

# plt.imshow(diff_final_avg)
# for h in [0, 10, 20, 30]:
#     plt.plot(diff_final_avg[:, h], label=f"h={h}")
# plt.legend()
for t in range(n_steps):
    plt.plot(diff_final_avg[t], label=f"t={t}")
plt.legend()
plt.show()

## Adaptive Time Schedule

In [ ]:
# adaptive time schedule
def compute_time_schedule(
    time: np.ndarray, delay: np.ndarray | None = None, C: int = 1, alpha: float = 1.0, T_min: float = 0.2
) -> np.ndarray:
    """
    time: (b, 1) or (a, b, 1)
    delay: (b,)
    return: (b, ah) or (a, b, ah)
    """
    idx = np.arange(50)[None, :]  # (1, ah)

    num_chunks = np.ceil((50 - delay) / C)  # (b,)
    chunk_idx = (idx - delay[:, None]) // C  # (b, ah), can be negative for positions < delay
    chunk_idx_valid = np.maximum(chunk_idx, 0)
    denom = np.maximum(num_chunks - 1, 1)[:, None]  # (b, 1)
    norm_idx = chunk_idx_valid / denom  # (b, ah)
    t_hit = T_min + (1 - T_min) * (norm_idx**alpha)  # (b, ah)

    if time.ndim == 2:
        time_schedule = 1 - (1 - time) / t_hit
    else:
        time_schedule = 1 - (1 - time) / t_hit[None, :, :]

    time_schedule = np.clip(time_schedule, 0.0, 1.0)
    return time_schedule


base_times = np.linspace(1, 0, n_steps + 1)[:, None, None]
t_schedule = compute_time_schedule(base_times, np.array(delay)[None], C=1, alpha=0.6, T_min=0.1)
dt_schedule = (t_schedule[1:] - t_schedule[:-1])[:, 0, delay:]  # (n_steps, ah-delay)
t_starts = t_schedule[:-1, 0, delay:]

## Estimated x1 analysis

In [ ]:
delta = t_starts[None, :, :, None] * v_t

x_1_hat = x_t[:, :-1] - delta

x_t_final = x_t[:, -1:]
diff_x1_hat = np.linalg.norm(x_1_hat - x_t_final, axis=-1)
diff_x1_hat_avg = np.mean(diff_x1_hat, axis=(0))

print(diff_x1_hat_avg.shape)

for t in range(n_steps):
    plt.plot(diff_x1_hat_avg[t], label=f"t={t}")
plt.legend()
plt.show()

## Straightness analysis

In [ ]:
def straightness(x, v, dt):
    d = x[:, :1] - x[:, -1:]  # x1-x0
    s = np.linalg.norm(d - v, axis=-1)
    print(s.shape)
    print(dt.shape)
    return np.sum(s * dt, axis=1)


stra = straightness(x_t, v_t, -dt_schedule)  # NOTE: dt_schedule is negative
stra_avg = np.mean(stra, axis=0)
plt.plot(stra_avg)
plt.show()

## Openloop plot

In [ ]:
# --- Prepare data for plotting ---
# Concatenate lists of arrays into single large arrays
if all_gt_actions:
    gt_actions_continuous = np.concatenate(all_gt_actions[:20], axis=0)
    inferred_actions_continuous = np.concatenate(all_infer_actions[:20], axis=0)

    total_steps, num_dims = gt_actions_continuous.shape
    time_steps_per_inference = gt_actions_continuous.shape[0] // 20

    # --- Plotting ---
    fig, axes = plt.subplots(7, 2, figsize=(20, 28), sharex=True)
    axes = axes.flatten()

    x_axis = np.arange(total_steps)

    for dim_idx in range(num_dims):
        ax = axes[dim_idx]

        # Plot the continuous action sequences
        ax.plot(x_axis, gt_actions_continuous[:, dim_idx], label="Ground Truth", color="cornflowerblue", alpha=0.9)
        ax.plot(
            x_axis, inferred_actions_continuous[:, dim_idx], label="Inferred", color="tomato", linestyle="--", alpha=0.9
        )

        # Mark the starting point of each inference sequence
        start_indices = np.arange(0, total_steps, time_steps_per_inference)
        ax.scatter(
            start_indices,
            gt_actions_continuous[start_indices, dim_idx],
            c="blue",
            marker="o",
            s=40,
            zorder=5,
            label="GT Start",
        )
        ax.scatter(
            start_indices,
            inferred_actions_continuous[start_indices, dim_idx],
            c="darkred",
            marker="x",
            s=40,
            zorder=5,
            label="Inferred Start",
        )

        ax.set_title(f"Action Dimension {dim_idx}")
        ax.set_ylabel("Value")
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend()

    # Set common X-axis label
    fig.supxlabel(f"Continuous Timestep (across {max_inferences} inferences)")

    plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust layout to make space for suptitle
    # fig.suptitle(f'Comparison of Ground Truth and Inferred Actions @Step {steps}', fontsize=18)
    # plt.savefig(f'{ckpt_root}/inferred_vs_gt_actions-{steps}.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No data was collected for plotting.")

In [ ]:
from tqdm import tqdm

max_inferences = 20
all_gt_actions = []
all_infer_actions = []
all_stream_actions = []
delay = 0

for i in tqdm(range(max_inferences)):
    data = dataset[i * config.model.action_horizon]  # i * chunk_size
    if i >= max_inferences:
        break

    gt_actions = data["actions"].numpy().copy()  # Shape: (50, 14)
    # del data["actions"]

    if delay > 0:
        data["action_prefix"] = gt_actions[:delay].copy()
        data["delay"] = np.array(delay)

    data["debug"] = True

    noise = np.random.normal(size=(50, 32))

    stream_actions = []

    def on_actions_ready(actions):
        stream_actions.append(actions)

    policy.infer_streaming(data.copy(), on_actions_ready=on_actions_ready, noise=noise)
    infer_actions = np.concatenate(stream_actions, axis=0)
    all_stream_actions.append(infer_actions)

    outputs = policy.infer(data.copy(), noise=noise)
    infer_actions = outputs["actions"]  # Shape: (50, 14)

    all_infer_actions.append(infer_actions[delay:])
    all_gt_actions.append(gt_actions)

# check whether all_stream_actions is the same as all_infer_actions
all_stream_actions = np.concatenate(all_stream_actions, axis=0)
all_infer_actions = np.concatenate(all_infer_actions, axis=0)

diff = all_stream_actions - all_infer_actions


In [ ]:
total_steps, num_dims = diff.shape

# --- Plotting ---
fig, axes = plt.subplots(7, 2, figsize=(20, 28), sharex=True)
axes = axes.flatten()

x_axis = np.arange(total_steps)

for dim_idx in range(num_dims):
    ax = axes[dim_idx]

    ax.plot(x_axis, diff[:, dim_idx], color="cornflowerblue", alpha=0.9)

    ax.set_title(f"Action Dimension {dim_idx}")
    ax.set_ylabel("Value")
    ax.grid(True, linestyle=":", alpha=0.6)

# Set common X-axis label
fig.supxlabel(f"Continuous Timestep (across {max_inferences} inferences)")

plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust layout to make space for suptitle
# fig.suptitle(f'Comparison of Ground Truth and Inferred Actions @Step {steps}', fontsize=18)
# plt.savefig(f'{ckpt_root}/inferred_vs_gt_actions-{steps}.png', dpi=300, bbox_inches='tight')
plt.show()